In [9]:
import os
import torch
from torch import nn
from skimage.transform import resize
from skimage.io import imread
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from tqdm.notebook import tqdm

Шаг 1. Загрузка и подготовка данных

1. Для начала мы скачаем датасет: [ADDI project](https://www.fc.up.pt/addi/ph2%20database.html).

<table>
    <tr>
        <td>
            <img src="PH2Dataset/PH2 Dataset images/IMD063/IMD063_Dermoscopic_Image/IMD063.bmp">
        </td>
        <td>
            <img src="PH2Dataset/PH2 Dataset images/IMD063/IMD063_lesion/IMD063_lesion.bmp">
        </td>
    </tr>
</table>

2. Разархивируем .rar файл.

Это фотографии двух типов **поражений кожи:** меланома и родинки.
В каждой папке IM D_00% находится информация об одном наблюдении. В подпапках соответствующего наблюдения - изображение в исходном виде и сегментированное изображение.

In [ ]:
!gdown 1T_RPkPP0jeWwK8L1UrmBw8V30eD7v6Ql

In [ ]:
get_ipython().system_raw("unrar x PH2Dataset.rar")

Изображения имеют разные размеры. Для однообразия и удобства работы создам кастомный класс для датасета с учетом размера изображений $256\times256 $ пикселей


In [11]:
class CustomDataset(TensorDataset):
    '''
        Some DOCstring need to be added
    '''
    
    def __init__(self,
                 root :str = 'PH2Dataset',
                 folder_name :str = 'PH2 Dataset images',
                 pic_size :tuple = (256, 256)):

        self.pic_size = pic_size
        self.images_paths = []
        self.lesions_paths = []

        # проходим по директориям и собираем пути к файлам
        for root, dirs, files in os.walk(os.path.join(root, folder_name)):
            if root.endswith(_Dermoscopic_Image):
                self.images_paths.append(os.path.join(root, files[0]))
            
                if root.endswith(_lesion):
                    self.lesions_paths.append(os.path.join(root, files[0]))
                
    def __len__(self):
        '''
        возращает количество изображений
        '''
        
        return len(self.images_paths)
        
    def __getitem__(self, idx):
        '''
        получаем очередное изображение -> трансформируем
        '''

        # transform.resize() для исходного изображения
        img_feature = imread(self.images_paths[idx])        
        img_feature = resize(img_feature,
                             self.pic_size,
                             mode=constant,
                             anti_aliasing=True,)

        # transform.resize() для сегментированного (обучающего) изображения
        img_target = imread(self.lesions_paths[idx])
        img_target = resize(img_target,
                            self.pic_size,
                            mode=constant,
                            anti_aliasing=False,) > 0.5
        
        return torch.from_numpy(
            np.array(
                np.rollaxis(
                    img_feature, 2, 0
                ), np.float32)
        ), torch.from_numpy(np.array(img_target, np.float32))